<a href="https://colab.research.google.com/github/rishi082000/Software-Hardware-CoDesign-for-Intelligent-Systems/blob/main/lenet5_cifar10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import argparse
import os, sys
import time
import datetime

# Import pytorch dependencies
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
from tqdm import tqdm_notebook as tqdm

# You cannot change this line.
from dataloader import CIFAR10

In [30]:
"""
Assignment 2(a)
Build the LeNet-5 model by following table 1 or figure 1.

You can also insert batch normalization and leave the LeNet-5
with batch normalization here for assignment 3(c).
"""
# Create the neural network module: LeNet-5
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()

        self.conv1 = nn.Conv2d(
            in_channels=3, # Changed from 1 to 3 to match CIFAR-10 RGB input
            out_channels=6,
            kernel_size=5,
            stride=1
        )
        # self.bn1 = nn.BatchNorm2d(6)

        self.conv2 = nn.Conv2d(
            in_channels=6,
            out_channels=16,
            kernel_size=5,
            stride=1
        )
        # self.bn2 = nn.BatchNorm2d(16)

        self.fc1 = nn.Linear(5 * 5 * 16, 120)
        # self.bn3 = nn.BatchNorm1d(120)

        self.fc2 = nn.Linear(120, 84)
        # self.bn4 = nn.BatchNorm1d(84)

        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):

        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, kernel_size=2, stride=2)

        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, kernel_size=2, stride=2)

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        # return F.softmax(x, dim=1)
        return x

In [31]:
"""
Hyperparameter optimization in assignment 4(a), 4(b) can be
conducted here.
Be sure to leave only your best hyperparameter combination
here and comment the original hyperparameter settings.
"""

# Setting some hyperparameters
TRAIN_BATCH_SIZE = 128
VAL_BATCH_SIZE = 100
INITIAL_LR = 0.1
MOMENTUM = 0.9
REG = 1e-4
EPOCHS = 30
DATAROOT = "./data"
CHECKPOINT_PATH = "./saved_model"

**Your answer:**

In [32]:
"""
Assignment 2(b)
Write functions to load dataset and preprocess the incoming data.
We recommend that the preprocess scheme \textbf{must} include
normalize, standardization, batch shuffling to make sure the training
process goes smoothly.
The preprocess scheme may also contain some data augmentation methods
(e.g., random crop, random flip, etc.).

Reference value for mean/std:

mean(RGB-format): (0.4914, 0.4822, 0.4465)
std(RGB-format): (0.2023, 0.1994, 0.2010)


NOTE: Considering this process has strong corrlelation with assignment 3(b),
please leave the data preprocessing method which can achieve the highest
validation accuracy here. You can include your original data augmentation
method as comments and denotes the accuracy difference between thest two
methods.
"""
# Specify preprocessing function.
# Reference mean/std value for
transform_train =  transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = (0.4914, 0.4822, 0.4464),
        std = (0.2023, 0.1994, 0.2010)
    )
])

transform_train_noaug = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean = (0.4914, 0.4822, 0.4464),
        std = (0.2023, 0.1994, 0.2010)
    )
])

transform_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean = (0.4914, 0.4822, 0.4465),
        std = (0.2023, 0.1994, 0.2010)
    )
])

**Your answer:**

In [33]:
# Call the dataset Loader
trainset = CIFAR10(root=DATAROOT, train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=1)
valset = CIFAR10(root=DATAROOT, train=False, download=True, transform=transform_val)
valloader = torch.utils.data.DataLoader(valset, batch_size=VAL_BATCH_SIZE, shuffle=False, num_workers=1)

Using downloaded and verified file: ./data/cifar10_trainval.tar.gz
Extracting ./data/cifar10_trainval.tar.gz to ./data
Files already downloaded and verified
Using downloaded and verified file: ./data/cifar10_trainval.tar.gz
Extracting ./data/cifar10_trainval.tar.gz to ./data
Files already downloaded and verified


In [34]:
# Specify the device for computation
device = 'cuda' if torch.cuda.is_available() else 'cpu'
net = LeNet5()
print(net)
net = net.to(device)
if device =='cuda':
    print("Train on GPU...")
else:
    print("Train on CPU...")

LeNet5(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)
Train on GPU...


In [35]:
# FLAG for loading the pretrained model
TRAIN_FROM_SCRATCH = False
# Code for loading checkpoint and recover epoch id.
CKPT_PATH = "./saved_model/model.h5"
def get_checkpoint(ckpt_path):
    try:
        ckpt = torch.load(ckpt_path)
    except Exception as e:
        print(e)
        return None
    return ckpt

ckpt = get_checkpoint(CKPT_PATH)
if ckpt is None or TRAIN_FROM_SCRATCH:
    if not TRAIN_FROM_SCRATCH:
        print("Checkpoint not found.")
    print("Training from scratch ...")
    start_epoch = 0
    current_learning_rate = INITIAL_LR
else:
    print("Successfully loaded checkpoint: %s" %CKPT_PATH)
    net.load_state_dict(ckpt['net'])
    start_epoch = ckpt['epoch'] + 1
    current_learning_rate = ckpt['lr']
    print("Starting from epoch %d " %start_epoch)

print("Starting from learning rate %f:" %current_learning_rate)

[Errno 2] No such file or directory: './saved_model/model.h5'
Checkpoint not found.
Training from scratch ...
Starting from learning rate 0.100000:


In [36]:
"""
Assignment 2(c)
In the targeted classification task, we use cross entropy loss with L2
regularization as the learning object.
You need to formulate the cross-entropy loss function in PyTorch.
You should also specify a PyTorch Optimizer to optimize this loss function.
We recommend you to use the SGD-momentum with an initial learning rate 0.01
and momentum 0.9 as a start.
"""
# Create loss function and specify regularization
criterion = nn.CrossEntropyLoss()
# Add optimizer
optimizer = optim.SGD(
    net.parameters(),
    lr = 0.1,
    momentum = 0.09,
    weight_decay = 5e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30   # number of epochs
)

In [37]:
"""
Assignment 3(a)
Start the training process over the whole CIFAR-10 training dataset.
For sanity check, you are required to report the initial loss value at
the beginning of the training process and briefly justify this value.
Run the training process for \textbf{a maximum of 30} epochs and you
should be able to reach around \textbf{65\%} accuracy on the validation
dataset.
"""
# Start the training/validation process
# The process should take about 5 minutes on a GTX 1070-Ti
# if the code is written efficiently.
global_step = 0
best_val_acc = 0

for i in range(start_epoch, EPOCHS):
    print(datetime.datetime.now())
    # Switch to train mode
    net.train()
    print("Epoch %d:" %i)

    total_examples = 0
    correct_examples = 0

    train_loss = 0
    train_acc = 0
    # Train the training dataset for 1 epoch.
    print(len(trainloader))
    for batch_idx, (inputs, targets) in enumerate(trainloader):
        # Copy inputs to device
        inputs = inputs.to(device)
        targets = targets.to(device)
        # Zero the gradient
        optimizer.zero_grad()
        # Generate output
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        # Now backward loss
        loss.backward()
        # Apply gradient
        optimizer.step()
        # Calculate predicted labels
        _, predicted = outputs.max(1)
        # Calculate accuracy
        total_examples += targets.size(0)
        correct_examples += predicted.eq(targets).sum().item()

        train_loss += loss

        global_step += 1
        if global_step % 100 == 0:
            avg_loss = train_loss / (batch_idx + 1)
        pass
    avg_acc = correct_examples / total_examples
    print("Training loss: %.4f, Training accuracy: %.4f" %(avg_loss, avg_acc))
    print(datetime.datetime.now())
    # Validate on the validation dataset
    print("Validation...")
    total_examples = 0
    correct_examples = 0

    net.eval()

    val_loss = 0
    val_acc = 0
    # Disable gradient during validation
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(valloader):
            # Copy inputs to device
            inputs = inputs.to(device)
            targets = targets.to(device)
            # Zero the gradient
            optimizer.zero_grad()
            # Generate output from the DNN.
            outputs = net(inputs)
            loss = criterion(outputs, targets)
            # Calculate predicted labels
            _, predicted = outputs.max(1)
            # Calculate accuracy
            total_examples += targets.size(0)
            correct_examples += predicted.eq(targets).sum().item()
            val_loss += loss.item()

    avg_loss = val_loss / len(valloader)
    avg_acc = correct_examples / total_examples
    print("Validation loss: %.4f, Validation accuracy: %.4f" % (avg_loss, avg_acc))


    """
    Assignment 4(b)
    Learning rate is an important hyperparameter to tune. Specify a
    learning rate decay policy and apply it in your training process.
    Briefly describe its impact on the learning curveduring your
    training process.
    Reference learning rate schedule:
    decay 0.98 for every 2 epochs. You may tune this parameter but
    minimal gain will be achieved.
    Assignment 4(c)
    As we can see from above, hyperparameter optimization is critical
    to obtain a good performance of DNN models. Try to fine-tune the
    model to over 70% accuracy. You may also increase the number of
    epochs to up to 100 during the process. Briefly describe what you
    have tried to improve the performance of the LeNet-5 model.
    """
    DECAY_EPOCHS = 2
    DECAY = 0.98
    if i % DECAY_EPOCHS == 0 and i != 0:
        current_learning_rate *= DECAY
        for param_group in optimizer.param_groups:
            # Assign the learning rate parameter
            param_group['lr'] = current_learning_rate
        print("Current learning rate has decayed to %f" %current_learning_rate)

    # Save for checkpoint
    if avg_acc > best_val_acc:
        best_val_acc = avg_acc
        if not os.path.exists(CHECKPOINT_PATH):
            os.makedirs(CHECKPOINT_PATH)
        print("Saving ...")
        state = {'net': net.state_dict(),
                 'epoch': i,
                 'lr': current_learning_rate}
        torch.save(state, os.path.join(CHECKPOINT_PATH, 'model.h5'))

    scheduler.step();

print("Optimization finished.")

2026-02-16 03:25:39.858224
Epoch 0:
352


<>:7: SyntaxWarning: invalid escape sequence '\%'
<>:7: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipython-input-2499778236.py:7: SyntaxWarning: invalid escape sequence '\%'
  should be able to reach around \textbf{65\%} accuracy on the validation


Training loss: 2.0297, Training accuracy: 0.2548
2026-02-16 03:25:57.806987
Validation...
Validation loss: 1.8506, Validation accuracy: 0.3140
Saving ...
2026-02-16 03:25:58.980567
Epoch 1:
352
Training loss: 1.6991, Training accuracy: 0.3781
2026-02-16 03:26:16.265081
Validation...
Validation loss: 1.5273, Validation accuracy: 0.4418
Saving ...
2026-02-16 03:26:17.737937
Epoch 2:
352
Training loss: 1.5755, Training accuracy: 0.4306
2026-02-16 03:26:35.441875
Validation...
Validation loss: 1.5971, Validation accuracy: 0.4236
Current learning rate has decayed to 0.098000
2026-02-16 03:26:36.657983
Epoch 3:
352
Training loss: 1.4914, Training accuracy: 0.4593
2026-02-16 03:26:54.218213
Validation...
Validation loss: 1.4575, Validation accuracy: 0.4716
Saving ...
2026-02-16 03:26:55.679073
Epoch 4:
352
Training loss: 1.4276, Training accuracy: 0.4869
2026-02-16 03:27:12.902717
Validation...
Validation loss: 1.5233, Validation accuracy: 0.4562
Current learning rate has decayed to 0.096040
